# Docling + Qdrant Advanced RAG Harness — Google Colab

This notebook demonstrates the same logical pipeline as the Docker harness without Docker-in-Docker. Select one of the three Docling chunkers: `hybrid`, `hierarchical`, or `line_based`.


In [ ]:
%pip install -qU "docling==2.127.0" "qdrant-client==1.19.0" "fastembed>=0.8,<1" "transformers>=4.55,<5"


## Configuration

Change `CHUNKER_TYPE` to compare chunking strategies. `hybrid` is the recommended general-purpose default.


In [ ]:
CHUNKER_TYPE = "hybrid"  # hybrid | hierarchical | line_based
TOKENIZER_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MAX_TOKENS = 120
DENSE_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
SPARSE_MODEL = "Qdrant/bm25"
BM25_LANGUAGE = "portuguese"
TOP_K = 5
CANDIDATE_K = 20


## Upload a document


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
filename = next(iter(uploaded))
path = Path('/content') / Path(filename).name
path.write_bytes(uploaded[filename])
print(path)


## Convert with Docling and apply the selected chunker


In [ ]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker, HierarchicalChunker
from docling_core.transforms.chunker.line_chunker import LineBasedTokenChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer

doc = DocumentConverter().convert(source=path).document

def tokenizer():
    hf = AutoTokenizer.from_pretrained(TOKENIZER_MODEL)
    return HuggingFaceTokenizer(tokenizer=hf, max_tokens=MAX_TOKENS)

if CHUNKER_TYPE == 'hierarchical':
    chunker = HierarchicalChunker(always_emit_headings=False)
elif CHUNKER_TYPE == 'line_based':
    chunker = LineBasedTokenChunker(tokenizer=tokenizer(), prefix='', omit_prefix_on_overflow=True)
else:
    chunker = HybridChunker(tokenizer=tokenizer(), merge_peers=True, repeat_table_header=True)

chunks = list(chunker.chunk(dl_doc=doc))
records = []
for i, chunk in enumerate(chunks):
    pages = sorted({int(prov.page_no) for item in getattr(chunk.meta, 'doc_items', []) for prov in getattr(item, 'prov', []) if getattr(prov, 'page_no', None) is not None})
    records.append({
        'index': i,
        'text': chunk.text,
        'context': chunker.contextualize(chunk=chunk),
        'headings': list(getattr(chunk.meta, 'headings', None) or []),
        'pages': pages,
    })
print('chunker=', CHUNKER_TYPE, 'chunks=', len(records))
records[:2]


## Embed and index in local Qdrant


In [ ]:
from fastembed import TextEmbedding, SparseTextEmbedding
from qdrant_client import QdrantClient, models
import uuid

dense_model = TextEmbedding(model_name=DENSE_MODEL)
sparse_model = SparseTextEmbedding(model_name=SPARSE_MODEL, language=BM25_LANGUAGE)
texts = [r['context'] for r in records]
dense = list(dense_model.passage_embed(texts))
sparse = list(sparse_model.embed(texts))

q = QdrantClient(path='/content/qdrant_local')
collection = 'documents'
if q.collection_exists(collection):
    q.delete_collection(collection)
q.create_collection(
    collection_name=collection,
    vectors_config={'dense': models.VectorParams(size=len(dense[0]), distance=models.Distance.COSINE)},
    sparse_vectors_config={'sparse': models.SparseVectorParams(modifier=models.Modifier.IDF)},
)
points=[]
for rec, dv, sv in zip(records, dense, sparse):
    points.append(models.PointStruct(
        id=str(uuid.uuid4()),
        vector={
            'dense': dv.tolist(),
            'sparse': models.SparseVector(indices=sv.indices.tolist(), values=sv.values.tolist()),
        },
        payload=rec,
    ))
q.upsert(collection_name=collection, points=points, wait=True)
print('indexed', len(points))


## Hybrid RRF retrieval


In [ ]:
def search(query: str, top_k=TOP_K, candidate_k=CANDIDATE_K):
    qd = list(dense_model.query_embed(query))[0]
    qs = list(sparse_model.query_embed(query))[0]
    response = q.query_points(
        collection_name=collection,
        prefetch=[
            models.Prefetch(query=qd.tolist(), using='dense', limit=candidate_k),
            models.Prefetch(query=models.SparseVector(indices=qs.indices.tolist(), values=qs.values.tolist()), using='sparse', limit=candidate_k),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k,
        with_payload=True,
    )
    return response.points

results = search('Qual é o principal assunto deste documento?')
for i, hit in enumerate(results, 1):
    p = hit.payload
    print(f"[S{i}] score={hit.score:.4f} pages={p.get('pages')} headings={p.get('headings')}")
    print(p.get('text','')[:500], '\n')


## Build an evidence block for an agent


In [ ]:
evidence=[]
for i, hit in enumerate(results, 1):
    p=hit.payload
    evidence.append(f"[S{i}] file={path.name} pages={p.get('pages')} headings={p.get('headings')}\n{p.get('context')}")
context='\n\n'.join(evidence)
print(context[:8000])
